# Chapter 25 — CTF 자동 풀이 플랫폼 (실습)

> **Chapter 25 · Part IV**

본문 §25.3 은 실제 CTFd 를 Docker Compose 로 격리 환경에 세우는 절차를 다룹니다. 이 노트북은 Docker 없이도 흐름을 확인할 수 있도록 CTFd 서버를 **인메모리 mock** 으로 재현합니다 — API 형태(요청/응답 구조)는 실제 CTFd 와 동일하게 맞췄습니다.

## 실습 목표
1. CTFd API 형태를 그대로 흉내 낸 mock 클라이언트 이해
2. CTF 챌린지 카테고리 분류기 구현
3. 카테고리별 도구 dispatch 구현
4. flag 추출 + CTFd 제출 루프 구현
5. CTFd 스코어보드 형태로 벤치마크 실행·해석


In [ ]:
# 환경 설정
import re, time
from collections import defaultdict


## Step 0. Mock CTFd 서버

실제로는 §25.3.3 의 `docker compose up` 으로 CTFd + MariaDB + Redis 를 띄우고, §25.3.4 의 `setup_challenges.py` 로 아래와 동일한 3 문제를 API 로 등록합니다. 이 노트북은 그 결과 상태를 메모리 안에서 재현합니다.

In [ ]:
class MockCTFdServer:
    """실제 CTFd 의 challenges/attempt API 응답 형태를 그대로 흉내낸다."""
    def __init__(self):
        self._challenges = {
            1: {'id': 1, 'name': 'Login Bypass', 'category': 'web',
                'description': 'Find the SQL injection in the login form at http://127.0.0.1:8081/login.',
                'flag': 'flag{sql1_1nj3ct10n}', 'solved': False},
            2: {'id': 2, 'name': 'Hidden Metadata', 'category': 'forensics',
                'description': 'This image file has a secret hidden in its metadata. See http://127.0.0.1:8082/file.',
                'flag': 'flag{ex1f_m3tadata}', 'solved': False},
            3: {'id': 3, 'name': 'Guess the Flag', 'category': 'misc',
                'description': 'No obvious hints here — read carefully and explore.',
                'flag': 'flag{misc_l4t3ral_th1nk1ng}', 'solved': False},
        }
        # 취약 서비스가 실제로 노출하는 evidence (Agent 가 공격해서 얻는 원문)
        self._evidence = {
            'http://127.0.0.1:8081/login': 'union select flag from secrets -- flag{sql1_1nj3ct10n}',
            'http://127.0.0.1:8082/file': 'file: JPEG image data\nExifTool: Comment: flag{ex1f_m3tadata}',
        }


## Step 1. CTFd API 클라이언트 (mock)

In [ ]:
class CTFdClient:
    """실제로는 requests 로 http://127.0.0.1:8000/api/v1/... 를 호출한다."""
    def __init__(self, server: MockCTFdServer):
        self.server = server

    def list_challenges(self) -> list:
        # TODO: 아래에 코드를 작성하세요.
# 힌트: return ...

    def get_challenge_detail(self, challenge_id: int) -> dict:
        # TODO: 아래에 코드를 작성하세요.
# 힌트: return ...

    def submit_flag(self, challenge_id: int, flag: str) -> dict:
        # TODO: 아래에 코드를 작성하세요.
# 힌트: chal = ...

server = MockCTFdServer()
client = CTFdClient(server)
print(client.list_challenges())


<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
        return ...
```
</details>


## Step 2. 챌린지 분류기

In [ ]:
CATEGORY_HINTS = {
    'web':       ['http', 'url', 'login', 'cookie', 'sql', 'xss'],
    'crypto':    ['cipher', 'rsa', 'xor', 'encrypt', 'modulus', 'key'],
    'forensics': ['pcap', 'image', 'memory dump', 'file', 'metadata'],
    'misc':      [],
}

def classify_challenge(name: str, description: str, category_hint: str = '') -> str:
    """CTFd 가 이미 category 필드를 주지만, 설명 텍스트로 재검증한다."""
    # TODO: 아래에 코드를 작성하세요.
# 힌트: if category_hint in CATEGORY_HINTS:

print(classify_challenge('Login Bypass', 'Find the SQL injection', 'web'))
print(classify_challenge('Guess the Flag', 'No obvious hints here'))


<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
    if category_hint in CATEGORY_HINTS:
        return ...
    text = ...
    scores = ...
        ...
        for cat, kws in CATEGORY_HINTS.items()
    ...
    best = ...
    return ...
```
</details>


## Step 3. Mock 도구 dispatch

In [ ]:
ALLOWED_TARGETS = {'127.0.0.1', 'localhost', 'chall-web'}  # 격리 lab 안의 자기 자산만

def run_web_strategy(target_url: str, srv: MockCTFdServer) -> dict:
    # TODO: 아래에 코드를 작성하세요.
# 힌트: host = ...

def run_forensics_strategy(target_url: str, srv: MockCTFdServer) -> dict:
    # TODO: 아래에 코드를 작성하세요.
# 힌트: evidence = ...

DISPATCH = {'web': run_web_strategy, 'forensics': run_forensics_strategy}
print(run_web_strategy('http://127.0.0.1:8081/login', server))


<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
    host = ...
    if host not in ALLOWED_TARGETS:
        return ...
    evidence = ...
    return ...
```
</details>


## Step 4. Flag 추출 + CTFd 제출 루프

In [ ]:
FLAG_PATTERN = re.compile(r'flag\{[^}]+\}', re.IGNORECASE)
MAX_ATTEMPTS = 5

def _extract_target(description: str) -> str:
    m = re.search(r'https?://[\w.\-:/]+', description)
    return m.group().rstrip('.,;:') if m else 'http://chall-web'  # 문장부호 오탐 방지

def solve_and_submit(client: CTFdClient, challenge_summary: dict) -> dict:
    """challenge_summary 는 client.list_challenges() 의 원소 하나."""
    # TODO: 아래에 코드를 작성하세요.
# 힌트: detail = ...

demo = solve_and_submit(client, client.list_challenges()[0])
print(demo)


<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
    detail = ...
    category = ...
    strategy = ...
    if strategy is None:
        return ...

    target = ...
    for attempt in range(1, MAX_ATTEMPTS + 1):
        result = ...
        evidence = ...
        match = ...
        if match:
            verdict = ...
            return ...
        ...

    return ...
```
</details>


## Step 5. CTFd 스코어보드 형태로 벤치마크

In [ ]:
def run_benchmark(client: CTFdClient) -> dict:
    # TODO: 아래에 코드를 작성하세요.
# 힌트: challenges = ...

# 새 서버·클라이언트로 처음부터 다시 (이전 Step 에서 이미 하나 풀었으므로)
server2 = MockCTFdServer()
client2 = CTFdClient(server2)
report = run_benchmark(client2)
print(f"전체 solve rate: {report['solve_rate']:.0%}")
for cat, stat in report['by_category'].items():
    print(f"  {cat}: {stat['solved']}/{stat['total']}")


<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
    challenges = ...
    results = ...
    solved = ...

    by_cat = ...
    for c, r in zip(challenges, results):
        cat = ...
        by_cat[cat]['total'] += ...
        if r['status'] == 'correct':
            by_cat[cat]['solved'] += ...

    return ...
        ...
```
</details>


## Step 6. 관찰 — 정적 규칙 Agent 의 한계

이 최소 구현은 web·forensics 두 카테고리만 실제로 풀립니다. misc 는 `DISPATCH` 에 전략이 없어 항상 `skipped` 로 끝납니다. 실제 CTFAgent·ctf-agent 류 도구는 이 지점에 LLM 기반 동적 전략 재수립을 추가해 훨씬 넓은 카테고리를 커버합니다 — 이 격차가 곧 본문 §25.5.2 가 말하는 "정적 규칙과 LLM reasoning의 능력 격차" 입니다.

실제 환경에서는 `MockCTFdServer` 대신 §25.3 에서 세운 진짜 CTFd 인스턴스에 `requests` 로 접속하면 이 노트북의 코드가 거의 그대로 동작합니다 — `CTFdClient` 의 메서드 시그니처를 본문 §25.4.1 의 실제 구현으로 바꾸기만 하면 됩니다.

---
## 정리

- 실제 CTFd 의 API 형태(list_challenges·get_challenge_detail·submit_flag)를 그대로 흉내낸 mock 클라이언트로 전체 흐름을 구현했습니다.
- CTF 챌린지 분류 → 카테고리별 도구 dispatch → flag 추출 → CTFd 제출까지 최소 기능 CTF Agent 를 구현했습니다.
- CTFd 스코어보드 형태로 solve rate 를 카테고리별로 측정했습니다.
- 정적 규칙 기반 구현의 한계(misc 미지원)를 직접 확인했습니다 — 다음 단계는 이 자리에 LLM 기반 전략을 추가해 ch27 Adaptive Defense 의 학습 신호로 재사용하는 것입니다.

> **다음 챕터:** ch26 AI vs AI 자동화 사이클과 Guardrail.